In [11]:
import json
import pandas as pd

metadata_tumor = pd.read_csv(
    "data/metadata/tcga_brca_tumor_one_per_patient.csv"
)

xena_clinical = pd.read_csv(
    "data/metadata/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix",
    sep="\t"
)

xena_clinical.shape

(1247, 194)

In [12]:
[
    col for col in xena_clinical.columns
    if any(term in col.lower() for term in ["pam50", "subtype", "intrinsic"])
]

['Integrated_Clusters_with_PAM50__nature2012',
 'PAM50Call_RNAseq',
 'PAM50_mRNA_nature2012',
 'SigClust_Intrinsic_mRNA_nature2012']

In [13]:
print(xena_clinical.head())
print(xena_clinical.columns[:10].tolist())

          sampleID AJCC_Stage_nature2012  \
0  TCGA-3C-AAAU-01                   NaN   
1  TCGA-3C-AALI-01                   NaN   
2  TCGA-3C-AALJ-01                   NaN   
3  TCGA-3C-AALK-01                   NaN   
4  TCGA-4H-AAAK-01                   NaN   

   Age_at_Initial_Pathologic_Diagnosis_nature2012  CN_Clusters_nature2012  \
0                                             NaN                     NaN   
1                                             NaN                     NaN   
2                                             NaN                     NaN   
3                                             NaN                     NaN   
4                                             NaN                     NaN   

  Converted_Stage_nature2012  Days_to_Date_of_Last_Contact_nature2012  \
0                        NaN                                      NaN   
1                        NaN                                      NaN   
2                        NaN                         

In [14]:
xena_clinical["PAM50Call_RNAseq"].value_counts(dropna=False)

PAM50Call_RNAseq
LumA      434
NaN       291
LumB      194
Basal     142
Normal    119
Her2       67
Name: count, dtype: int64

In [15]:
print(metadata_tumor.shape)
print(metadata_tumor["sample_barcode"].head())

(1095, 9)
0    TCGA-3C-AAAU-01A
1    TCGA-3C-AALI-01A
2    TCGA-3C-AALJ-01A
3    TCGA-3C-AALK-01A
4    TCGA-4H-AAAK-01A
Name: sample_barcode, dtype: object


In [17]:
metadata_tumor["sample_id_15"] = metadata_tumor["sample_barcode"].str[:15]

xena_clinical["sample_id_15"] = xena_clinical["sampleID"].str[:15]

In [18]:
metadata_tumor["sample_id_15"].nunique(), metadata_tumor.shape[0]

(1095, 1095)

In [19]:
xena_clinical["sample_id_15"].nunique(), xena_clinical.shape[0]

(1247, 1247)

In [20]:
pam50 = xena_clinical[
    ["sample_id_15", "PAM50Call_RNAseq"]
].copy()

In [21]:
pam50["sample_id_15"].duplicated().sum()

0

In [22]:
metadata_tumor_annotated = metadata_tumor.merge(
    pam50,
    on="sample_id_15",
    how="left",
    validate="one_to_one"
)

In [25]:
print(metadata_tumor.shape)
print(metadata_tumor_annotated.shape)

metadata_tumor_annotated[
    "PAM50Call_RNAseq"
].value_counts(dropna=False)

(1095, 10)
(1095, 11)


PAM50Call_RNAseq
LumA      421
NaN       253
LumB      192
Basal     139
Her2       67
Normal     23
Name: count, dtype: int64

In [26]:
metadata_tumor_annotated[
    metadata_tumor_annotated["PAM50Call_RNAseq"] == "Normal"
][["sample_barcode"]].head(10)

,sample_barcode
8,TCGA-A1-A0SB-01A
33,TCGA-A2-A0CL-01A
46,TCGA-A2-A0CZ-01A
87,TCGA-A2-A0YK-01A
98,TCGA-A2-A1G6-01A
100,TCGA-A2-A25A-01A
205,TCGA-A8-A08H-01A
261,TCGA-AC-A2FK-01A
341,TCGA-AO-A03R-01A
342,TCGA-AO-A03T-01A


In [27]:
metadata_tumor_annotated.to_csv(
    "data/metadata/tcga_brca_tumor_metadata_annotated.csv",
    index=False
)